# **The 2nd URA Hackathon 2026**

## Installation

In [ ]:
import os

def install_if_missing(import_name, pip_command):
    """Install a package via pip only if it cannot already be imported.

    Args:
        import_name: Module name used for the import check (e.g. "cv2").
        pip_command: Argument string passed to `pip install` when the module is missing.
    """
    try:
        __import__(import_name)
        print(f"'{import_name}' Existed")
    except ImportError:
        print(f"Installing: {pip_command}")
        from IPython import get_ipython
        ipython = get_ipython()
        if ipython is not None:
            ipython.run_line_magic('pip', f'install {pip_command}')
        else:
            os.system(f'pip install {pip_command}')
install_if_missing("cv2", "opencv-python")
install_if_missing("paddleocr", "paddleocr==2.8.0 paddlepaddle")
install_if_missing("vietocr", "vietocr")
install_if_missing("rapidfuzz", "rapidfuzz")
install_if_missing("underthesea", "underthesea")

In [ ]:
# 1. Standard Library
import os
import re
import csv
import time
import unicodedata
from collections import Counter

# 2. Data Processing & Math
import numpy as np
import pandas as pd

# 3. Image Processing
import cv2
import PIL
from PIL import Image, ImageEnhance, ImageFilter

# 4. Machine Learning
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.model_selection import RandomizedSearchCV

# 5. OCR & NLP
# PaddleOCR (Text Detection)
from paddleocr import PaddleOCR

# VietOCR (Text Recognition)
from vietocr.tool.predictor import Predictor
from vietocr.tool.config import Cfg

# Underthesea (Named Entity Recognition for Vietnamese)
from underthesea import ner

# 6. Utilities
from tqdm.notebook import tqdm

# 7. Environment Bug Fixes
if not hasattr(PIL._util, 'is_directory'):
    PIL._util.is_directory = os.path.isdir
    
if not hasattr(np, 'sctypes'):
    np.sctypes = {
        "float": [np.float16, np.float32, np.float64],
        "int": [np.int8, np.int16, np.int32, np.int64],
        "uint": [np.uint8, np.uint16, np.uint32, np.uint64],
        "complex": [np.complex64, np.complex128]
    }

print('All libraries loaded')

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


Success


## Brand Rules (Regex Dictionary)

In [ ]:
# BRAND_RULES: (regex, canonical_name, [product_line_keywords])
BRAND_RULES = [
    # NAN 
    (r"(?:sua|milk|bot|pha).*\bnan\b|\bnan\b.*(?:sua|milk|bot|opti|infini|supreme|formula|pha)", "Nestlé Nan", [
        (r"infini(?:\s*pro)?(?:\s*a2)?", "InfiniPro A2"),
        (r"opti\s*pro\s*plus|optiproplus", "Optipro Plus"),
        (r"opti\s*pro|optipro", "Optipro"),
        (r"supreme", "Supreme")
    ]),
    
    # Ha Long Canfoco & Đồ Hộp Hạ Long
    (r"ha\s*long\s*canfoco|halong\s*canfoco|canfood|canfoco", "Ha Long Canfoco", [
        (r"pate.*c[ộo]t|c[ộo]t\s*đ[èeềếêi]n|c[ộo]t\s*d[âa]n", "Pate Cột Đèn"),
        (r"pate", "Pate"),
        (r"cá\s*ngừ|ca\s*ngu", "Cá Ngừ")
    ]),
    (r"đ[ồo]\s*h[ộo]p\s*h[ạa]\s*long|do\s*hop\s*ha\s*long", "Đồ Hộp Hạ Long", [
        (r"pate.*c[ộo]t|c[ộo]t\s*đ[èeềếêi]n|c[ộo]t\s*d[âa]n", "Patê Cột Đèn")
    ]),
    (r"(?:pate|do\s*hop|thit\s*hop|ca\s*hop|cot\s*den).*ha\s*long|ha\s*long.*(?:pate|do\s*hop|thit\s*hop|ca\s*hop|cot\s*den)", "Ha Long Canfoco", [
        (r"pate.*c[ộo]t|c[ộo]t\s*đ[èe]n|c[ộo]t\s*d[âa]n", "Pate Cột Đèn"),
        (r"pate", "Pate")
    ]),
    
    # Pate Cột Đèn Hải Phòng - placed before generic Pate
    (r"c[ộo]t\s*đ[èe]n|cot\s*den|c[ộo]t\s*d[âa]n", "Pate Cột Đèn Hải Phòng", []),
    
    # Vinamilk
    (r"vinamilk", "Vinamilk", [
        (r"flex", "Flex"),
        (r"adm(?:\s*gold)?", "ADM Gold"),
        (r"sure\s*prevent|sure", "Sure Prevent"),
        (r"canxi", "Canxi Pro"),
        (r"colos\s*baby|colosbaby", "ColosBaby"),
        (r"ông\s*thọ|ong\s*tho", "Ông Thọ"),
        (r"dielac(?:\s*grow(?:\s*plus)?)?", "Dielac Grow Plus"),
        (r"grow\s*plus", "Grow Plus"),
        (r"grow", "Grow")
    ]),
    
    # TH True Milk
    (r"th true|thtrue", "TH True Milk", [
        (r"yogurt|sữa\s*chua", "True Yogurt"),
        (r"grow", "Grow"),
        (r"school", "School Milk"),
        (r"butter|bơ", "True Butter")
    ]),
    
    # Dutch Lady
    (r"dutch lady|cô gái", "Dutch Lady", [
        (r"grow", "Grow"),
        (r"complete", "Complete"),
        (r"canxi", "Canxi"),
        (r"mom", "Mom")
    ]),
    
    # Nutifood
    (r"nutifood|nuti\b", "Nutifood", [
        (r"grow\s*plus|growplus", "Grow Plus"),
        (r"pedia", "Pedia Plus"),
        (r"iq", "IQ Gold")
    ]),
    
    # Ensure
    (r"ensure\b", "Abbott Ensure", [
        (r"gold", "Gold"),
        (r"original", "Original"),
        (r"plus", "Plus")
    ]),
    
    # Other Abbott
    (r"pediasure", "Abbott PediaSure", []),
    (r"similac", "Abbott Similac", []),
    (r"glucerna", "Abbott Glucerna", []),
    
    # Milo
    (r"milo\b", "Nestlé Milo", []),
    
    # Nestlé General
    (r"nestle|nestlé", "Nestlé", [
        (r"milo", "Milo"),
        (r"coffee\s*mate", "Coffee Mate"),
        (r"carnation", "Carnation"),
        (r"nestea", "Nestea"),
        (r"nan", "Nan")
    ]),
    
    # Aptamil
    (r"aptamil", "Aptamil", []),
    
    # Friso
    (r"friso\b", "Friso", [
        (r"gold", "Gold"),
        (r"comfort", "Comfort"),
        (r"prestige", "Prestige")
    ]),
    
    # Meiji
    (r"meiji\b", "Meiji", [
        (r"growing", "Growing Up"),
        (r"step", "Step")
    ]),
    
    # Ba Vì
    (r"ba vi\b|bavi\b|ba vì", "Ba Vì", [
        (r"gold", "Gold")
    ]),
    
    # Lothamilk
    (r"lothamilk", "Lothamilk", []),
    
    # Yomost
    (r"yomost", "Yomost", []),
    
    # Đà Lạt Milk
    (r"\b(d[àa]\s*l[ạa]t|dalat)(?:\s*milk)?\b", "Đà Lạt Milk", []),
    
    # Kun
    (r"kun\b|kun milk", "Kun", [
        (r"choco", "Chocolate"),
        (r"straw", "Strawberry")
    ]),
    
    # Fami
    (r"fami\b", "Fami", [
        (r"canxi", "Canxi"),
        (r"kid", "Kid")
    ]),
    
    # Anlene
    (r"anlene", "Anlene", [
        (r"gold", "Gold"),
        (r"concentrate", "Concentrate")
    ]),
    
    # Anchor
    (r"anchor\b", "Anchor", [
        (r"butter|bơ", "Butter"),
        (r"cream|kem", "Cream")
    ]),
    
    # Vissan
    (r"vissan", "Vissan", [
        (r"pate\s*heo", "Pate Heo"),
        (r"pate\s*ga|pate\s*gà", "Pate Gà"),
        (r"xúc\s*xích|xuc\s*xich", "Xúc Xích"),
        (r"lạp\s*xưởng|lap\s*xuong", "Lạp Xưởng")
    ]),
    
    # Hafi
    (r"hafi\b", "Hafi", []),
    
    # Ba Huân
    (r"ba huan|ba huân", "Ba Huân", []),
    
    # San Hà
    (r"san ha\b|san hà", "San Hà", []),
    
    # CP
    (r"(?:\bcp\b|c\.p\.)(?=.*(?:pate|xuc\s*xich|thit|ga|heo|food))|(?:pate|xuc\s*xich|thit).*(?:\bcp\b|c\.p\.)", "CP", [
        (r"pate", "Pate"),
        (r"xúc\s*xích|xuc\s*xich", "Xúc Xích Heo")
    ]),
    
    # Long Biên
    (r"long bien|long biên", "Long Biên", []),
    
    # Fallback Pate
    (r"\bpate\b|patê", "Pate", []),
    
    # Coffee
    (r"highlands?\s*coffee", "Highlands Coffee", []),
    (r"phúc\s*long|phuc\s*long", "Phúc Long", []),
    (r"the\s*coffee\s*house|coffee\s*house", "The Coffee House", []),
    
    # Baby Formula (Others)
    (r"hipp\s*combiotic|hipp\s*organic|hipp\b", "HiPP", [
        (r"combiotic", "Combiotic"),
        (r"organic", "Organic")
    ]),
    (r"illuma\b", "ILLUMA", []),
    (r"beba\b", "Nestlé BEBA", []),
    
    # Canned food (Others)
    (r"quang\s*h[oồ]ng?\s*sardine|quang\s*hong", "Quang Hong Sardine", []),
    (r"3\s*bông\s*mai|ba\s*bông\s*mai|3\s*bong\s*mai", "3 Bông Mai", [
        (r"pate", "Pate Gan")
    ]),
    (r"expect\s*pate|expect\b", "Expect Pate", []),
    (r"nhân\s*h[oò]a|nhan\s*hoa", "Nhân Hòa Foods", []),
    
    # Condiments
    (r"chin[\-\s]*su|chinsu", "Chin-Su", [
        (r"tương\s*ớt|tuong\s*ot", "Tương Ớt")
    ]),
    
    # Cosmetics
    (r"acnes\b", "Acnes", [
        (r"vitamin\s*cleanser", "Vitamin Cleanser")
    ]),
    # ============ PHASE 2 EXPANSION PACK ============
    # Cosmetics & Personal Care
    (r"dove\b", "Dove", [
        (r"smoothie", "Smoothie tẩy da chết"),
    ]),
    (r"l.oreal|loreal|l'oreal", "L'Oreal", []),
    (r"olay\b", "Olay", []),
    (r"klairs", "Klairs", []),
    (r"torriden", "Torriden", [
        (r"dive\s*in", "Dive In"),
    ]),
    (r"hada\s*labo", "Hada Labo", []),
    (r"hatomugi", "Hatomugi", []),
    (r"mixa\b", "Mixa", []),
    (r"d'alba|d\salba|dalba", "d'Alba", []),
    (r"cosmettes", "Cosmettes", []),
    (r"beplain", "beplain", []),
    (r"colugea", "Colugea", []),
    (r"biore\b", "Biore", []),
    (r"laroche|la\s*roche", "La Roche-Posay", []),
    (r"cetaphil", "Cetaphil", []),
    (r"bioderma", "Bioderma", []),
    (r"barbie\s*skin", "Barbie Skin", []),
    (r"embryolisse", "Embryolisse", []),
    (r"ziaja", "ZIAJA", []),
    (r"skin1004", "Skin1004", []),
    (r"propolis\s*essence", "Propolis", []),
    (r"herbalife", "Herbalife", []),
    (r"lorsia", "Lorsia", []),
    (r"sắc\s*ngọc\s*khang|sac\s*ngoc\s*khang", "Sắc Ngọc Khang", []),

    # New Milks & Foods
    (r"oggi\b", "VitaDairy", [
        (r"gold", "OGGI Gold"),
        (r"", "OGGI"),
    ]),
    (r"vita\s*dairy|vitadairy", "VitaDairy", [
        (r"colos\s*baby|colosbaby", "ColosBaby"),
    ]),
    (r"colos\s*baby|colosbaby", "VitaDairy", [
        (r"gold", "ColosBaby Gold"),
        (r"pedia", "ColosBaby Pedia"),
        (r"", "ColosBaby"),
    ]),
    (r"custas", "Custas", []),
    (r"burine", "Burine", []),
    (r"huggies", "Huggies", []),
    (r"lineabon", "LineaBon", []),
    (r"kingbaby", "KingBaby", []),
    (r"binggrae|ginggrae", "Binggrae", []),
    (r"natureone|nature\s*one", "NatureOne", []),
    (r"unicity", "Unicity", []),
    (r"traphaco", "Traphaco", []),
    (r"sinocare", "Sinocare", []),
    (r"vf\s*gold|vfgold", "VF Gold", []),
]







## Load Dataset

In [ ]:
import kagglehub
import os

# Download latest version
path = kagglehub.competition_download('the-2nd-ura-hackathon')

print("Path to competition files:", path)

DATA_PATHS = {}

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        filepath = os.path.join(dirname, filename)
        # Automatically find the CSV files
        if filename == 'private_test.csv':
            DATA_PATHS['test_csv'] = filepath
        elif filename == 'train_labels.csv':
            DATA_PATHS['train_label_csv'] = filepath
            
        # Automatically find the image folders!
        elif filename.endswith('.jpg'):
            if 'phase2' in dirname.lower() and 'test_images' not in DATA_PATHS:
                DATA_PATHS['test_images'] = dirname

print("Successfully mapped dynamic paths:")
for key, path in DATA_PATHS.items():
    print(f" - {key}: {path}")

Path to competition files: /kaggle/input/competitions/the-2nd-ura-hackathon


In [ ]:
import pandas as pd
train_labels = pd.read_csv(DATA_PATHS['train_label_csv'], keep_default_na=False)
print('Success')

## Preprocessing and Postprocessing

In [ ]:
def preprocess(img_pil, max_dim=1536, min_dim=800):
    """Prepare a thumbnail image for OCR text detection.

    Steps: rescale tiny thumbnails up / huge images down, boost local contrast with
    CLAHE on the lightness channel only (preserves colour), apply a mild sharpen, and
    return an RGB numpy array ready for PaddleOCR.

    Args:
        img_pil: Source image as a PIL.Image.
        max_dim: Long side is downscaled to this if larger.
        min_dim: Long side is upscaled to this if smaller.

    Returns:
        np.ndarray (H, W, 3) in RGB order.
    """
    img = np.array(img_pil.convert('RGB'))[:, :, ::-1]
    h, w = img.shape[:2]
    
    # Smart Rescaling (Upscale tiny thumbnails, downscale massive images)
    if max(w, h) < min_dim:
        ratio = min_dim / max(w, h)
        img = cv2.resize(img, (int(w * ratio), int(h * ratio)), interpolation=cv2.INTER_CUBIC)
    elif max(w, h) > max_dim:
        ratio = max_dim / max(w, h)
        img = cv2.resize(img, (int(w * ratio), int(h * ratio)), interpolation=cv2.INTER_AREA)

    # Convert to LAB color space to apply CLAHE to the lightness channel only (preserves colors)
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    cl = clahe.apply(l)
    
    limg = cv2.merge((cl,a,b))
    img = cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)
    
    # Mild sharpening
    kernel = np.array([[0, -0.5, 0], 
                       [-0.5, 3, -0.5], 
                       [0, -0.5, 0]])
    img = cv2.filter2D(img, -1, kernel)
    
    # Return as RGB Numpy array for PaddleOCR
    return img[:, :, ::-1]


def postprocess_ocr(text: str) -> str:
    """Normalize whitespace and drop consecutive duplicate tokens (common in thumbnails)."""
    text = re.sub(r"\s+", " ", text).strip()
    tokens = text.split()
    if not tokens:
        return ""
    deduped = [tokens[0]]
    for tok in tokens[1:]:
        if tok.lower() != deduped[-1].lower():
            deduped.append(tok)
    return " ".join(deduped)

print('Success')

## Text Detection using PaddleOCR

In [ ]:
import torch
Detector = PaddleOCR(
    use_angle_cls=True, 
    lang='en', 
    det=True, 
    rec=False, 
    use_gpu=torch.cuda.is_available(), 
    show_log=False, 
    det_db_box_thresh=0.3,
    det_db_unclip_ratio=2.0, 
    det_limit_side_len=1536    
)

print('Success')


## Text Recognition using VietOCR

In [ ]:
import torch
config = Cfg.load_config_from_name('vgg_transformer')
config['device'] = 'cuda:0' if torch.cuda.is_available() else 'cpu'

recognizer = Predictor(config)

18533it [00:19, 957.09it/s] 


## Crop + Padding + Sort

In [ ]:
def Crop_Padding(image, bbox, pad=5):
    """Crop the axis-aligned bounding rectangle of `bbox` from `image`.

    Args:
        image: Source image as an (H, W, C) numpy array.
        bbox: Polygon points [[x, y], ...] of a detected text box.
        pad: Extra padding in pixels added around the box (clamped to image bounds).

    Returns:
        The cropped image region as a numpy array.
    """
    box = np.array(bbox, dtype=int)
    x_min = max(0, np.min(box[:, 0]) - pad)
    x_max = min(image.shape[1], np.max(box[:, 0]) + pad)
    y_min = max(0, np.min(box[:, 1]) - pad)
    y_max = min(image.shape[0], np.max(box[:, 1]) + pad)
    
    return image[y_min:y_max, x_min:x_max]

def Sort_Boxes(boxes):
    """Order detection boxes in natural reading order (top-to-bottom, left-to-right).

    Boxes are grouped into lines using a vertical threshold derived from the median
    box height; each line is then sorted left-to-right.

    Args:
        boxes: List of polygon boxes, each as [[x, y], ...] with 4 corner points.

    Returns:
        The boxes reordered for reading.
    """
    if not boxes:
        return []        
    # Initial vertical sort
    boxes = sorted(boxes, key=lambda box: box[0][1]) # Sort top-left coord
    threshold = np.median([abs(b[2][1] - b[0][1]) for b in boxes]) * 0.3
    sorted_boxes = []
    current_line = [boxes[0]]
    baseline_y = boxes[0][0][1] # Top-left y coord of the first box

    for box in boxes[1:]:
        # If the box is within the vertical threshold of the line's baseline
        if abs(box[0][1] - baseline_y) <= threshold:
            current_line.append(box)
        else:
            # Line completed: sort the line horizontally 
            sorted_boxes.extend(sorted(current_line, key=lambda b: b[0][0]))
            # Start a new line
            current_line = [box]
            baseline_y = box[0][1]

    # Commit the final remaining line
    if current_line:
        sorted_boxes.extend(sorted(current_line, key=lambda b: b[0][0]))

    return sorted_boxes

print('Success')

## ML Fallback Model

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline, FeatureUnion
import numpy as np

from sklearn.model_selection import RandomizedSearchCV

class ProductPredictor:
    """Lightweight, text-only product head trained on OCR text.

    Two TF-IDF (char + word n-grams) + LogisticRegression stages:
      - a "noise gate" (`is_product`) predicting whether any product is present, and
      - a multi-class classifier (`predict`) naming the product.
    A regex rule function (passed to `fit`) takes priority over the ML output.
    """

    def __init__(self, min_class_count=1, prob_threshold=0.50, auto_tune=False):
        self.min_class_count = min_class_count
        self.prob_threshold = prob_threshold
        self.auto_tune = auto_tune
        self._has_clf = self._prod_clf = None
        self._n_train = self._n_classes = 0

    def fit(self, train_labels, rule_fn):
        """Fit the noise gate and the product classifier.

        Args:
            train_labels: DataFrame with `ocr_text` and `product_name` columns.
            rule_fn: Regex rule function used at predict time before the ML head.
        """
        df = train_labels.copy()
        df["ocr_text"] = df["ocr_text"].astype(str).str.strip()
        df["product_name"] = df["product_name"].astype(str).str.strip()
        self._rule_fn = rule_fn
        
        # --- NOISE GATE: binary "does this text mention a product?" ---
        union_has_clf = FeatureUnion([
            ("char_features", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=3000, min_df=2)),
            ("word_features", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), max_features=1500, min_df=2))
        ])
        self._has_clf = Pipeline([
            ("union", union_has_clf),
            ("clf", LogisticRegression(C=1.0, max_iter=400, class_weight="balanced")),
        ])
        self._has_clf.fit(df["ocr_text"], (df["product_name"] != "").astype(int))
        
        # --- PRODUCT CLASSIFIER: multi-class product name ---
        pos = df[(df["ocr_text"] != "") & (df["product_name"] != "")]
        keep = pos["product_name"].value_counts()
        keep_all = keep[keep >= self.min_class_count].index
        pos_all = pos[pos["product_name"].isin(keep_all)]
        
        if not len(pos_all):
            return self

        union_prod_clf = FeatureUnion([
            ("char_features", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=3000, min_df=2)),
            ("word_features", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), max_features=1500, min_df=2))
        ])
        self._prod_clf = Pipeline([
            ("union", union_prod_clf),
            ("clf", LogisticRegression(C=1.0, max_iter=400, class_weight="balanced")),
        ])
        
        self._prod_clf.fit(pos_all["ocr_text"], pos_all["product_name"])
            
        self._n_train = len(df)
        self._n_classes = pos_all["product_name"].nunique()
        return self

    def predict(self, ocr_text):
        ocr_text = "" if ocr_text is None else str(ocr_text).strip()
        if not ocr_text or self._prod_clf is None:
            return ""
        return str(self._prod_clf.predict([ocr_text])[0])
    def is_product(self, ocr_text):
        if not ocr_text or self._has_clf is None:
            return False
        proba = self._has_clf.predict_proba([ocr_text])[0]
        prod_idx = list(self._has_clf.classes_).index(1)
        return proba[prod_idx] >= self.prob_threshold

    
print('Success')

## Core Extraction Pipeline

In [ ]:
import re
import unicodedata


# ==================== UTILITIES ====================

def normalize_vi(s: str) -> str:
    return unicodedata.normalize("NFD", s).encode("ascii", "ignore").decode().lower()

def brands_match(a: str, b: str) -> bool:
    a, b = normalize_vi(a), normalize_vi(b)
    return a == b or a in b or b in a


# ==================== LAYER 1: REGEX ====================

def extract_product(text: str) -> tuple:
    if not text or not text.strip():
        return (" ", " ")
    tl = text.lower()
    tl_norm = normalize_vi(text)
    is_e = "patê" in tl

    for pattern, brand, lines in BRAND_RULES:
        if re.search(pattern, tl_norm, re.IGNORECASE):
            if lines:
                for line_pat, canonical_line in lines:
                    if line_pat and re.search(line_pat, tl_norm, re.IGNORECASE):
                        prod = canonical_line.replace("Pate", "Patê").replace("pate", "patê") if is_e else canonical_line
                        return (brand, prod)
            return (brand, " ")
    return (" ", " ")

def split_prediction(combined_str: str) -> tuple:
    if not combined_str or not combined_str.strip():
        return (" ", " ")
    combined_str = combined_str.strip()
    known_brands = sorted(
        set(b for _, b, _ in BRAND_RULES),
        key=len, reverse=True
    )
    for brand in known_brands:
        if normalize_vi(combined_str).startswith(normalize_vi(brand.lower())):
            prod = combined_str[len(brand):].strip()
            return (brand, prod if prod else " ")
    return (" ", combined_str)

def safe_ml_predict(text: str) -> tuple | None:
    try:
        raw = product_predictor.predict(text)
        if not raw or not isinstance(raw, str):
            return None
        brand, prod = split_prediction(raw)
        if len(brand) < 2 or len(brand) > 60:
            return None
        if brand.isdigit() or prod.isdigit():
            return None
        return (brand.strip(), prod.strip())
    except Exception:
        return None

def extract_product_by_subtraction(ocr_text, brand):
    if not brand or brand == " ":
        return " "
    cleaned = re.sub(re.escape(brand), "", ocr_text, flags=re.IGNORECASE).strip()
    cleaned = re.sub(r'\d+[gG](?:\s|$)', ' ', cleaned)
    cleaned = re.sub(r'\d+[mM][lL]', ' ', cleaned)
    cleaned = re.sub(r'\d+[kK](?:g|G)', ' ', cleaned)
    cleaned = re.sub(r'@\S+', ' ', cleaned)
    cleaned = re.sub(r'#\S+', ' ', cleaned)
    cleaned = re.sub(r'\d{10,}', ' ', cleaned)
    cleaned = re.sub(r'₫|VND|\$', ' ', cleaned)
    words = [w for w in cleaned.split() if len(w) >= 2]
    product = " ".join(words[:5])
    return product if product else " "


# ==================== LAYER 2: NER ====================

_NOISE_TOKEN = re.compile(r'^(#|\d+ml|\d+g|\d+l|\d+kg|₫|\$|%)', re.IGNORECASE)
_NER_BLACKLIST = {
    # Vietnamese common words
    "me", "mẹ", "bộ", "ban", "anh", "em", "ông", "bà", "chị", "bé",
    "con", "các", "của", "cho", "và", "hay", "với", "từ", "tại",
    "gia", "nhà", "hội", "sở", "phòng", "ban",
    # English common words  
    "ing", "the", "and", "for", "new", "hot", "top", "all", "get",
    "mini", "love", "comb", "set", "box", "mix", "pro", "max",
}

def ner_extract_brand(ocr_text: str) -> tuple | None:
    try:
        entities = ner(ocr_text)
        brand_parts = []
        product_parts = []
        collecting_brand = False
        brand_done = False

        for word, pos, chunk, ner_tag in entities:
            if ner_tag == "B-ORG":
                collecting_brand = True
                brand_done = False
                brand_parts.append(word)
            elif ner_tag == "I-ORG" and collecting_brand:
                brand_parts.append(word)
            else:
                if collecting_brand:
                    collecting_brand = False
                    brand_done = True
                if brand_done:
                    if _NOISE_TOKEN.match(word):
                        break
                    if len(word) >= 2:
                        product_parts.append(word)

        if brand_parts:
            brand = " ".join(brand_parts).strip()
            product = " ".join(product_parts[:4]).strip()
            if len(brand) >= 3 and not brand.isdigit() and brand.lower() not in _NER_BLACKLIST:
                return (brand, product if product else " ")
        return None
    except Exception:
        return None


# ==================== NOISE GATES ====================

PRODUCT_KEYWORDS = [
    # Cosmetics & Skincare
    "kem", "serum", "dưỡng", "da", "tẩy trang", "tẩy da chết", "sữa rửa mặt", 
    "toner", "lotion", "chống nắng", "mặt nạ", "mask", "son", "môi", "mụn", 
    "thâm", "nám", "tóc", "gội", "xả", "cleanser", "cream", "body", "ẩm", "trắng",
    
    # Milk & Baby
    "sữa", "milk", "bột", "bé", "trẻ", "sơ sinh", "tã", "bỉm", "canxi", "dha",
    "chua", "yogurt", "pha sẵn", "công thức", "dinh dưỡng",
    
    # Food
    "xúc xích", "lạp xưởng", "pate", "đồ hộp", "thịt", "heo", "bò", "gà", "cá",
    
    # Other household
    "khăn", "giấy", "ướt", "tã", "bỉm"
]

def text_has_product_keyword(text: str) -> bool:
    if not text:
        return False
    text_lower = text.lower()
    
    for kw in PRODUCT_KEYWORDS:
        # \b ensures it only matches the exact word (e.g. "da", not "đang")
        if re.search(r'\b' + re.escape(kw) + r'\b', text_lower):
            return True
            
    return False


# ==================== LAYER 3: SPATIAL ====================

def extract_brand_from_boxes(box_data):
    if not box_data or len(box_data) < 3:
        return None
    ranked = sorted(box_data, key=lambda b: b["area"], reverse=True)
    avg_area = sum(b["area"] for b in box_data) / len(box_data)
    if ranked[0]["area"] < avg_area * 1.5:
        return None
    text = ranked[0]["text"].strip()
    if len(text) < 2 or len(text) > 30:
        return None
    if text.isdigit() or text.startswith("@") or text.startswith("#"):
        return None
    return text

def extract_product_from_boxes(box_data, brand_text):
    if not box_data or len(box_data) < 2:
        return " "
    ranked = sorted(box_data, key=lambda b: b["area"], reverse=True)
    for box in ranked:
        text = box["text"].strip()
        if text.lower() == brand_text.lower():
            continue
        if len(text) < 2 or len(text) > 40:
            continue
        if text.isdigit() or text.startswith("@") or text.startswith("#"):
            continue
        return text
    return " "


# ==================== ML SETUP ====================

product_predictor = ProductPredictor(min_class_count=1, prob_threshold=0.50)
product_predictor.fit(train_labels, extract_product)


# ==================== ORCHESTRATOR ====================

def predict_product(ocr_text: str, box_data: list = None) -> tuple:
    if not ocr_text or str(ocr_text).strip().lower() in ["", "nan", "none", "null", "na"]:
        return (" ", " ")
    regex_brand, regex_prod = extract_product(ocr_text)

    # 1. Regex
    if regex_brand.strip():
        if not regex_prod.strip():
            ml_result = safe_ml_predict(ocr_text)
            if ml_result:
                ml_brand, ml_prod = ml_result
                if brands_match(regex_brand, ml_brand) and ml_prod.strip():
                    return (regex_brand, ml_prod)
            sub_prod = extract_product_by_subtraction(ocr_text, regex_brand)
            return (regex_brand, sub_prod)
        return (regex_brand, regex_prod)

    # 2. NER
    ner_result = ner_extract_brand(ocr_text)
    if ner_result and text_has_product_keyword(ocr_text):
        brand, prod = ner_result
        if not prod.strip():
            prod = extract_product_by_subtraction(ocr_text, brand)
        return (brand, prod)

    # 3. Spatial
    box_brand = extract_brand_from_boxes(box_data)
    if box_brand and text_has_product_keyword(ocr_text):
        prod = extract_product_from_boxes(box_data, box_brand)
        return (box_brand, prod)

    # 4. Fallback
    return (" ", " ")

## Main Loop

In [ ]:
import os
import csv
import pandas as pd
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm

test_df = pd.read_csv(DATA_PATHS['test_csv'])
test_image_folder = DATA_PATHS['test_images']

results = []
checkpoint_path = "submission_checkpoint.csv"
SAVE_EVERY = 100  

for index, row in tqdm(test_df.iterrows(), total=len(test_df)):
    file_name = row['image_id']  
    image_path = os.path.join(test_image_folder, file_name + '.jpg')
    
    ocr_text = " "
    brand_name = " "
    product_name = " "
    
    if not os.path.exists(image_path):
        results.append({
            "image_id": file_name,
            "ocr_text": " ",
            "brand_name": " ",
            "product_name": " "
        })
        continue
        
    try:
        img = Image.open(image_path).convert("RGB")
        img = preprocess(img)
        
        result = Detector.ocr(img, cls=False)
        
        if result is None or len(result) == 0 or result[0] is None:
            ocr_text = " "
        else:
            boxes = [line[0] for line in result[0]]
            
            if boxes:
                boxes = Sort_Boxes(boxes)
                texts = []
                box_data = []
                for x in boxes:
                    crop = Crop_Padding(img, x, pad=8)
                    if crop.shape[0] < 8 or crop.shape[1] < 8:
                        continue
                    crop_pil = Image.fromarray(crop)
                    text = recognizer.predict(crop_pil)
                    if len(text.strip()) > 1:
                        texts.append(text)
                        width = abs(x[1][0] - x[0][0])
                        height = abs(x[2][1] - x[1][1])
                        box_data.append({"text": text.strip(), "area": width * height})

                ocr_text = " ".join(texts)
                ocr_text = postprocess_ocr(ocr_text)
                brand_name, product_name = predict_product(ocr_text, box_data)
                
    except Exception as e:
        pass
        
    if not ocr_text or str(ocr_text).strip() == "" or str(ocr_text).lower() in ["nan", "none", "null", "na"]:
        ocr_text = " "
        
    if not brand_name or str(brand_name).strip() == "":
        brand_name = " "

    if not product_name or str(product_name).strip() == "":
        product_name = " "

    results.append({
        "image_id": file_name,
        "ocr_text": ocr_text,
        "brand_name": brand_name,
        "product_name": product_name
    })
    
    if (index + 1) % SAVE_EVERY == 0:
        pd.DataFrame(results).to_csv(checkpoint_path, index=False)

final_df = pd.DataFrame(results, columns=['image_id', 'ocr_text', 'brand_name', 'product_name'])
final_df.to_csv("submission.csv", index=False)


## Postprocessing

In [ ]:
import csv

final_df = pd.DataFrame(results, columns=['image_id', 'ocr_text', 'brand_name', 'product_name'])

for col in ['ocr_text', 'brand_name', 'product_name']:
    final_df[col] = final_df[col].astype(str)
    final_df.loc[final_df[col].str.strip() == "", col] = " "
    final_df.loc[final_df[col].str.lower().isin(["nan", "none", "null", "na"]), col] = " "

final_df.to_csv("submission.csv", index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)
print("Done!")

Done!
